# Дистилляция модели и CFG: большой teacher → компактный student

Здесь **обучается новая меньшая модель с нуля**. Teacher — замороженные EMA-веса
вращающихся лучей из `demo/pipeline` (128 каналов, 4 residual blocks). Student —
96 каналов, 2 блока. Teacher объединяет conditional и unconditional predictions;
student учится выдавать этот результат одним forward при фиксированном `GUIDANCE=1.5`.
15% null-condition примеров сохраняют отдельную unconditional ветку student.

Идея выходной дистилляции guided teacher опирается на
[On Distillation of Guided Diffusion Models, §3.1 и Appendix B.2](https://arxiv.org/html/2210.03142v3#S3.SS1).
В работе guidance подаётся отдельным входом, student инициализируется teacher,
а следующий этап сокращает шаги. Здесь — фиксированный guidance, более компактная
архитектура и random initialization. Это учебный эксперимент output KD,
architecture compression и guidance baking; оба sampler сохраняют **100 DDIM steps**.

В статье используется `(1+w)*conditional - w*unconditional`; в нашем API
`unconditional + scale*(conditional-unconditional)`, то есть `scale=1+w`.
Веса и пакет скачиваются отдельными файлами из GitHub. Все формулы и обучение
видны ниже; функции trainer/fit не используются. Нужны Python 3.11+ и интернет
для первой установки. По умолчанию обучение выполняется на CPU.

In [ ]:
import os
import subprocess
import sys
REPOSITORY_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
# Служебный флаг для локальной проверки уже установленного checkout.
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "diffusion-lab[interactive] @ git+" + REPOSITORY_URL,
    ])

In [ ]:
import copy
import hashlib
import importlib.metadata
import json
import platform
import tempfile
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from diffusers import DDIMScheduler
from IPython.display import Image, Markdown, display
from diffusion_lab import PointDenoiser, ResearchDiffusionPipeline
from diffusion_lab.environments import make_environment
from diffusion_lab.metrics import evaluate_samples, sliced_wasserstein
from diffusion_lab.schedulers import prediction_to_x0
from diffusion_lab.visualization import (
    notebook_field_explorer, notebook_sampling_explorer, plot_dataset,
    plot_rainbow_trajectories, save_figure, save_trajectory_gif,
)
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

PROJECT_ROOT = next((root for root in (Path.cwd(), *Path.cwd().parents)
                     if (root / "demo/pipeline/model_index.json").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="guided-kd-")) / "repository"
    subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_ROOT)])
TEACHER_PATH = PROJECT_ROOT / "demo/pipeline"
print("Teacher:", TEACHER_PATH)

## Параметры исследования

Меняйте архитектуру, MSE / дополнительные loss terms, dropout, optimizer и loop.
`GUIDANCE` определяет training target. Чтобы получить student с другим baked
scale, его нужно обучить заново. `INFERENCE_STEPS` выбирается одинаковым у всех
моделей; этот notebook не обучает уменьшение числа шагов. Для короткой проверки
можно уменьшить `EPOCHS`, `TRAIN_SAMPLES`, `EVAL_SAMPLES` и `MAKE_GIFS`.

In [ ]:
DEVICE = "cpu"                    # также "cuda" / "mps", если доступны
TORCH_THREADS = 4
TRAIN_SEED = 801
EVAL_SEED = 802                   # отдельные точки, шум и timestep для parity
SAMPLING_SEED = 803               # одинаковый начальный шум всех sampler
EPOCHS = 150
TRAIN_SAMPLES = 4096
REFERENCE_SAMPLES = 4096
EVAL_SAMPLES = 1024
BATCH_SIZE = 256
STUDENT_HIDDEN = 96
STUDENT_LAYERS = 2
GUIDANCE = 1.5
NULL_PROBABILITY = 0.15
X0_LOSS_WEIGHT = 0.1
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
INFERENCE_STEPS = 100
TIMING_REPEATS = 2
INCLUDE_TEACHER_CFG_OFF = True
MAKE_GIFS = True
GIF_PARTICLES = 12
GIF_MAX_FRAMES = 50
OUTPUT_DIR = "outputs/model_distillation"

In [ ]:
if EPOCHS < 1 or not 0 <= NULL_PROBABILITY < 1 or X0_LOSS_WEIGHT < 0:
    raise ValueError("Check EPOCHS, NULL_PROBABILITY and X0_LOSS_WEIGHT")
if GUIDANCE < 0 or not np.isfinite(GUIDANCE):
    raise ValueError("GUIDANCE must be finite and nonnegative")
torch.set_num_threads(TORCH_THREADS)
torch.manual_seed(TRAIN_SEED)
device = torch.device(DEVICE)
RUN_DIR = Path(OUTPUT_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
parent = ResearchDiffusionPipeline.from_pretrained(
    TEACHER_PATH, local_files_only=True, low_cpu_mem_usage=False,
).to(device)
teacher = parent.model.eval().requires_grad_(False)
train_scheduler = DDIMScheduler.from_config(parent.scheduler.config)
assert train_scheduler.config.prediction_type == "v_prediction"
T = train_scheduler.config.num_train_timesteps
assert T == 1000 and 1 <= INFERENCE_STEPS <= T
num_classes = teacher.config.num_classes
train_data = make_environment("rotating_rays", n_samples=TRAIN_SAMPLES,
                              n_classes=num_classes, seed=TRAIN_SEED)
reference = make_environment("rotating_rays", n_samples=REFERENCE_SAMPLES,
                             n_classes=num_classes, seed=EVAL_SEED)
loader_generator = torch.Generator().manual_seed(TRAIN_SEED + 10)
# CPU generators support CPU/MPS; random tensors are transferred explicitly.
noise_generator = torch.Generator().manual_seed(TRAIN_SEED + 11)
loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                    generator=loader_generator)
student = PointDenoiser(
    sample_shape=teacher.sample_shape, hidden_dim=STUDENT_HIDDEN,
    num_layers=STUDENT_LAYERS, conditioner="embedding", num_classes=num_classes,
).to(device)
untrained_student = copy.deepcopy(student).eval().requires_grad_(False)
optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE,
                              weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.1,
)
parameter_count = lambda model: sum(parameter.numel() for parameter in model.parameters())
print(f"Teacher: {parameter_count(teacher):,}; student: {parameter_count(student):,} parameters")
print(f"Parameter compression: {parameter_count(teacher) / parameter_count(student):.2f}×")

def model_state_sha256(model):
    digest = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        digest.update(name.encode("utf-8"))
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

teacher_state_before = model_state_sha256(teacher)
assert teacher.sample_shape == train_data.sample_shape


## Target и явный PyTorch loop

\[
v_{KD}=v_u+s(v_c-v_u),\qquad x_0=\sqrt{\bar\alpha_t}x_t-\sqrt{1-\bar\alpha_t}v.
\]

Если `keep_condition=False`, target заменяется на `v_u`, а student получает
выученный null token через `condition_mask`. Teacher не получает градиентов.
Основной loss — MSE по v; дополнительный MSE по teacher x0 включён с весом 0.1.
Для v parameterization этот x0 loss перевзвешивает ту же ошибку множителем
`1-alpha_bar_t`; это не независимый ground-truth supervision term.

In [ ]:
@torch.no_grad()
def teacher_target(noisy, timesteps, conditions, keep_condition, guidance=GUIDANCE):
    teacher.eval()
    v_unconditional = teacher(noisy, timesteps, conditions=None).sample
    v_conditional = teacher(noisy, timesteps, conditions=conditions).sample
    guided = v_unconditional + guidance * (v_conditional - v_unconditional)
    return torch.where(keep_condition[:, None], guided, v_unconditional)

loss_fn = nn.MSELoss()
history = []
start_training = time.perf_counter()
for epoch in range(EPOCHS):
    student.train()
    totals = {"loss": 0.0, "v_mse": 0.0, "x0_mse": 0.0, "null_fraction": 0.0}
    count = 0
    for batch in loader:
        clean = batch["samples"].to(device)
        conditions = batch["conditions"].to(device)
        noise = torch.randn(clean.shape, generator=noise_generator).to(device)
        timesteps = torch.randint(T, (len(clean),), generator=noise_generator).to(device)
        keep_condition = (torch.rand(len(clean), generator=noise_generator)
                          >= NULL_PROBABILITY).to(device)
        noisy = train_scheduler.add_noise(clean, noise, timesteps)
        target = teacher_target(noisy, timesteps, conditions, keep_condition)
        optimizer.zero_grad(set_to_none=True)
        prediction = student(noisy, timesteps, conditions=conditions,
                             condition_mask=keep_condition).sample
        v_loss = loss_fn(prediction, target)
        x0_student = prediction_to_x0(train_scheduler, noisy, prediction, timesteps)
        x0_teacher = prediction_to_x0(train_scheduler, noisy, target, timesteps)
        x0_loss = loss_fn(x0_student, x0_teacher)
        loss = v_loss + X0_LOSS_WEIGHT * x0_loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite distillation loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        for name, value in (("loss", loss), ("v_mse", v_loss), ("x0_mse", x0_loss),
                            ("null_fraction", (~keep_condition).float().mean())):
            totals[name] += float(value.detach()) * len(clean)
        count += len(clean)
    lr_scheduler.step()
    row = {"epoch": epoch + 1, **{name: value / count for name, value in totals.items()},
           "lr": optimizer.param_groups[0]["lr"]}
    history.append(row)
    if epoch == 0 or (epoch + 1) % 25 == 0 or epoch + 1 == EPOCHS:
        print(f"epoch {epoch+1:3d}/{EPOCHS}: KD={row['v_mse']:.5f}, x0={row['x0_mse']:.5f}")
training_seconds = time.perf_counter() - start_training
student.eval()
assert all(parameter.grad is None for parameter in teacher.parameters())
print(f"Training completed in {training_seconds:.1f}s; teacher remained frozen")
teacher_state_after = model_state_sha256(teacher)
assert teacher_state_before == teacher_state_after, "Frozen teacher changed"


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for key in ("loss", "v_mse", "x0_mse"):
    axes[0].semilogy([row["epoch"] for row in history],
                     [max(row[key], 1e-12) for row in history], label=key)
axes[0].set(xlabel="Epoch", ylabel="KD error", title="Actual student training")
axes[0].legend(frameon=False)
axes[1].plot([row["epoch"] for row in history], [row["null_fraction"] for row in history])
axes[1].axhline(NULL_PROBABILITY, color="black", linestyle="--", linewidth=1)
axes[1].set(xlabel="Epoch", ylabel="Fraction", title="Observed null-condition training")
save_figure(fig, RUN_DIR / "training_loss")
display(fig)
plt.close(fig)

## Проверка регрессии на новых точках и новом шуме

Parity сопоставляет **предсказания** student и guided teacher на одинаковых x_t;
оно не заменяет проверку генерации. Ошибка по времени помогает найти участки,
где student плохо аппроксимирует teacher. Null MSE проверяет отдельно сохранённую
unconditional ветку. Обучающая и контрольная выборки имеют разные seed.

In [ ]:
probe_generator = torch.Generator().manual_seed(EVAL_SEED + 20)
probe_clean = reference.samples.to(device)
probe_conditions = reference.conditions.to(device)
probe_noise = torch.randn(probe_clean.shape, generator=probe_generator).to(device)
probe_timesteps = torch.randint(T, (len(probe_clean),), generator=probe_generator).to(device)
probe_noisy = train_scheduler.add_noise(probe_clean, probe_noise, probe_timesteps)
with torch.no_grad():
    target = teacher_target(probe_noisy, probe_timesteps, probe_conditions,
                            torch.ones(len(probe_clean), device=device, dtype=torch.bool))
    predicted = student(probe_noisy, probe_timesteps, conditions=probe_conditions).sample
    teacher_null = teacher(probe_noisy, probe_timesteps, conditions=None).sample
    student_null = student(probe_noisy, probe_timesteps, conditions=None).sample
    v_mse = float(loss_fn(predicted, target))
    null_mse = float(loss_fn(student_null, teacher_null))
    x0_mse = float(loss_fn(prediction_to_x0(train_scheduler, probe_noisy, predicted, probe_timesteps),
                           prediction_to_x0(train_scheduler, probe_noisy, target, probe_timesteps)))
parity_metrics = {"heldout_v_mse": v_mse, "heldout_x0_mse": x0_mse,
                  "heldout_null_v_mse": null_mse}
print(parity_metrics)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
expected, actual = target.cpu().numpy().ravel(), predicted.cpu().numpy().ravel()
colors = np.repeat(probe_timesteps.cpu().numpy(), train_data.sample_shape[0])
indices = np.linspace(0, len(expected) - 1, min(3000, len(expected)), dtype=int)
artist = axes[0].scatter(expected[indices], actual[indices], c=colors[indices],
                         cmap="viridis", s=5, alpha=0.45)
low, high = min(expected.min(), actual.min()), max(expected.max(), actual.max())
axes[0].plot([low, high], [low, high], "k--", linewidth=1)
axes[0].set(xlabel="Guided teacher v", ylabel="Student v", title=f"Held-out parity · MSE={v_mse:.4g}")
fig.colorbar(artist, ax=axes[0], label="Training timestep t")
error = (predicted - target).square().mean(-1).cpu().numpy()
times = probe_timesteps.cpu().numpy()
centers, binned = [], []
for left in range(0, T, max(1, T // 10)):
    right = min(T, left + max(1, T // 10))
    selected = (times >= left) & (times < right)
    if selected.any():
        centers.append((left + right - 1) / 2)
        binned.append(float(error[selected].mean()))
axes[1].plot(centers, binned, "o-")
axes[1].set(xlabel="timestep bin", ylabel="Mean v error", title="KD error across noise levels")
save_figure(fig, RUN_DIR / "teacher_student_parity")
display(fig)
plt.close(fig)

## Сравнение генерации и вычислений

Все модели используют один официальный **DDIM grid**, одинаковые labels, initial
noise seed и число шагов. Student запускается с `guidance_scale=1`: scale 1.5 уже
встроен в его условные веса. Дополнительный CFG student снова смешивает его
conditional и unconditional predictions и **повторно усиливает guidance**.

NFE считает реальные вызовы model.forward hooks. Teacher при scale 1.5 делает
два forward на шаг, student — один. Время — медиана повторов после прогрева,
включая одинаковую запись истории; это измерение данного устройства и batch,
а не универсальная оценка ускорения. SWD сравнивается с отдельными реальными
точками; teacher-student SWD измеряет сохранение распределения teacher.

In [ ]:
labels = torch.arange(EVAL_SAMPLES) % num_classes

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()

def timed_sample(model, guidance_scale, conditions=labels, repeats=TIMING_REPEATS):
    pipeline = ResearchDiffusionPipeline(model, DDIMScheduler.from_config(parent.scheduler.config))
    generator_device = device if device.type == "cuda" else "cpu"
    # Equal small warm-up for all models, excluded from measurement.
    pipeline(batch_size=8, conditions=conditions[:8] if conditions is not None else None,
             num_inference_steps=min(5, INFERENCE_STEPS), guidance_scale=guidance_scale,
             generator=torch.Generator(device=generator_device).manual_seed(SAMPLING_SEED))
    elapsed, nfe = [], []
    for repeat in range(repeats):
        calls = [0]
        def count_forward(*args):
            calls[0] += 1
        hook = model.register_forward_hook(count_forward)
        synchronize()
        started = time.perf_counter()
        try:
            output = pipeline(batch_size=EVAL_SAMPLES, conditions=conditions,
                              num_inference_steps=INFERENCE_STEPS, guidance_scale=guidance_scale,
                              generator=torch.Generator(device=generator_device).manual_seed(SAMPLING_SEED),
                              record_history=True)
            synchronize()
            elapsed.append(time.perf_counter() - started)
            nfe.append(calls[0])
        finally:
            hook.remove()
    assert len(set(nfe)) == 1
    return output, float(np.median(elapsed)), nfe[0]

models = [("Teacher guided", teacher, GUIDANCE),
          ("Student untrained", untrained_student, 1.0),
          ("Student distilled", student, 1.0)]
if INCLUDE_TEACHER_CFG_OFF:
    models.append(("Teacher CFG off (conditional)", teacher, 1.0))
results, rows = {}, []
for name, model, scale in models:
    output, seconds, nfe = timed_sample(model, scale)
    results[name] = output
    metrics = evaluate_samples(reference, output.samples, labels, seed=EVAL_SEED)
    rows.append({"model": name, "parameters": parameter_count(model), "guidance_scale": scale,
                 "steps": INFERENCE_STEPS, "nfe": nfe, "median_seconds": seconds, **metrics})
    print(f"{name}: SWD={metrics['sliced_wasserstein']:.5f}, NFE={nfe}, median={seconds:.3f}s")
teacher_output = results["Teacher guided"]
student_output = results["Student distilled"]
assert torch.equal(teacher_output.timesteps, student_output.timesteps)
assert torch.equal(teacher_output.history[0], student_output.history[0])
assert rows[0]["nfe"] == INFERENCE_STEPS * (1 if GUIDANCE == 1 else 2)
assert rows[2]["nfe"] == INFERENCE_STEPS
for row in rows:
    row["teacher_student_swd"] = sliced_wasserstein(
        teacher_output.samples.cpu().numpy(), results[row["model"]].samples.cpu().numpy(), seed=EVAL_SEED)
columns = ["model", "parameters", "nfe", "median_seconds", "sliced_wasserstein", "teacher_student_swd"]
columns += [f"class_{i}_sliced_wasserstein" for i in range(num_classes)]
header = "| " + " | ".join(columns) + " |"
separator = "| " + " | ".join(["---"] * len(columns)) + " |"
lines = [header, separator]
for row in rows:
    lines.append("| " + " | ".join(f"{row[key]:.4g}" if isinstance(row[key], float)
                                  else str(row[key]) for key in columns) + " |")
display(Markdown("\n".join(lines)))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4), constrained_layout=True)
plot_dataset(reference.samples[:EVAL_SAMPLES], reference.conditions[:EVAL_SAMPLES],
             ax=axes[0], title="Held-out data", size=7)
for axis, row in zip(axes[1:], rows[:3]):
    plot_dataset(results[row["model"]].samples, labels, ax=axis, size=7,
                 title=f"{row['model']}\nSWD={row['sliced_wasserstein']:.4f} · NFE={row['nfe']}")
for axis in axes:
    axis.set(xlim=(-2.5, 2.5), ylim=(-2.5, 2.5))
save_figure(fig, RUN_DIR / "generation_comparison")
display(fig)
plt.close(fig)

## Реальный денойзинг: trajectories и GIF

История берётся из actual DDIM steps без повторного шума. Каждый цвет привязан
к исходному времени пути. Start — шум, Finish — итоговая точка; это траектории
денойзинга, а не физическая задача A→B. Colormap по умолчанию `turbo_r`.

In [ ]:
trajectory_files = []
for name, output in (("teacher", teacher_output), ("student", student_output)):
    paths = output.history[:, :GIF_PARTICLES].permute(1, 0, 2)
    fig, ax = plot_rainbow_trajectories(paths, cmap="turbo_r", title=f"{name} · real DDIM steps",
                                       start_label="Noise x_T", end_label="Sample x_0")
    trajectory_files += [str(path) for path in save_figure(fig, RUN_DIR / f"{name}_trajectories")]
    display(fig)
    plt.close(fig)
    if MAKE_GIFS:
        gif = save_trajectory_gif(paths, RUN_DIR / f"{name}_trajectories.gif", cmap="turbo_r",
                                  max_frames=GIF_MAX_FRAMES, fps=12,
                                  title=f"{name} · noise → sample", start_label="Noise x_T", end_label="Sample x_0")
        trajectory_files.append(str(gif))
        display(Image(filename=str(gif)))

## Unconditional и интерактивный student

Null branch действительно обучалась на target `v_unconditional`. Она не
является классом 0. Ниже обе модели получают `conditions=None` и scale 1.
В interactive student выключен внешний **Use CFG**: встроенный `GUIDANCE`
определяется training target, а не этим slider.

In [ ]:
teacher_unconditional, _, _ = timed_sample(teacher, 1.0, conditions=None, repeats=1)
student_unconditional, _, _ = timed_sample(student, 1.0, conditions=None, repeats=1)
null_metrics = {
    "teacher": evaluate_samples(reference, teacher_unconditional.samples, seed=EVAL_SEED),
    "student": evaluate_samples(reference, student_unconditional.samples, seed=EVAL_SEED),
}
print("Unconditional distribution metrics:", null_metrics)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
plot_dataset(teacher_unconditional.samples, ax=axes[0], title="Teacher · unconditional", size=7)
plot_dataset(student_unconditional.samples, ax=axes[1], title="Student · trained null branch", size=7)
save_figure(fig, RUN_DIR / "unconditional_comparison")
display(fig)
plt.close(fig)
student_pipeline = ResearchDiffusionPipeline(student, DDIMScheduler.from_config(parent.scheduler.config))
fields = notebook_field_explorer(student_pipeline, reference, resolution=24, timestep=349)
fields.controls["cfg_enabled"].value = False
fields.controls["condition"].value = 1
fields.controls["potential"].value = True
fields.controls["potential_log"].value = True
sampling = notebook_sampling_explorer(student_pipeline, reference, num_samples=256,
                                       steps=INFERENCE_STEPS, seed=SAMPLING_SEED, guidance_scale=1.0)
sampling.controls["cfg_enabled"].value = False
sampling.controls["condition"].value = 1
display(fields, sampling)

## Сохранить student, training state и результаты

Экспорт содержит **raw student** без EMA и обычный Diffusers scheduler. Отдельный
checkpoint хранит optimizer, LR scheduler и состояния noise / DataLoader / torch
RNG; manifest фиксирует baked guidance, архитектуру, seed, метрики и teacher SHA256.
Для воспроизводимого продолжения используйте те же данные и hyperparameters,
восстановите optimizer/LR/RNG и продолжайте видимый loop с сохранённого epoch.
Загрузка ниже проверяет совпадение однопроходной генерации. Дополнительный CFG
при deployment оставляйте выключенным (`guidance_scale=1`).

In [ ]:
student_pipeline.save_pretrained(RUN_DIR / "student_pipeline", safe_serialization=True)
fixed_guidance_config = {
    "fixed_guidance": GUIDANCE, "inference_guidance": 1.0,
    "num_inference_steps": INFERENCE_STEPS,
    "null_prediction": "trained to teacher unconditional v_prediction",
}
(RUN_DIR / "student_pipeline/distillation_config.json").write_text(
    json.dumps(fixed_guidance_config, indent=2), encoding="utf-8",
)
checkpoint = {
    "student": student.state_dict(), "optimizer": optimizer.state_dict(),
    "lr_scheduler": lr_scheduler.state_dict(), "epoch": EPOCHS,
    "history": history, "noise_rng": noise_generator.get_state(),
    "loader_rng": loader_generator.get_state(), "torch_rng": torch.random.get_rng_state(),
    "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
    "baked_guidance": GUIDANCE, "null_probability": NULL_PROBABILITY,
}
torch.save(checkpoint, RUN_DIR / "training_checkpoint.pt")
np.savez_compressed(RUN_DIR / "generated_samples.npz", labels=labels.numpy(),
                    teacher=teacher_output.samples.cpu().numpy(),
                    student=student_output.samples.cpu().numpy(),
                    untrained=results["Student untrained"].samples.cpu().numpy())
weights_path = TEACHER_PATH / "model/diffusion_pytorch_model.safetensors"
manifest = {
    "method": "v_prediction output KD + architecture compression + fixed guidance baking",
    "paper": "https://arxiv.org/abs/2210.03142", "teacher_sha256": hashlib.sha256(weights_path.read_bytes()).hexdigest(),
    "teacher_state_sha256_before": teacher_state_before,
    "teacher_state_sha256_after": teacher_state_after,
    "teacher_architecture": dict(teacher.config), "student_architecture": dict(student.config),
    "baked_guidance": GUIDANCE, "deployment_guidance_scale": 1.0,
    "null_probability": NULL_PROBABILITY, "x0_loss_weight": X0_LOSS_WEIGHT,
    "epochs": EPOCHS, "optimizer_steps": EPOCHS * len(loader), "training_seconds": training_seconds,
    "train_samples": TRAIN_SAMPLES, "reference_samples": REFERENCE_SAMPLES,
    "eval_samples": EVAL_SAMPLES, "train_seed": TRAIN_SEED, "eval_seed": EVAL_SEED,
    "sampling_seed": SAMPLING_SEED, "inference_steps": INFERENCE_STEPS,
    "timing_repeats": TIMING_REPEATS, "timing_includes_history": True,
    "device": str(device), "threads": TORCH_THREADS, "platform": platform.platform(),
    "torch_version": torch.__version__, "diffusers_version": importlib.metadata.version("diffusers"),
    "diffusion_lab_version": importlib.metadata.version("diffusion-lab"),
    "parity": parity_metrics, "generation_comparison": rows,
    "unconditional": null_metrics, "trajectory_files": trajectory_files,
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(RUN_DIR / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
reloaded = ResearchDiffusionPipeline.from_pretrained(
    RUN_DIR / "student_pipeline", local_files_only=True, low_cpu_mem_usage=False,
).to(device)
reloaded_output, _, reloaded_nfe = timed_sample(reloaded.model, 1.0, repeats=1)
torch.testing.assert_close(reloaded_output.samples, student_output.samples, rtol=0, atol=0)
assert reloaded_nfe == INFERENCE_STEPS
print("Saved and verified:", RUN_DIR.resolve())
print("Use pipeline(..., guidance_scale=1.0) for the baked-guidance student")